# load packages

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score
import sys
import numpy as np
from sklearn.linear_model import LogisticRegression
import argparse as ap
import time
import inspect
from datetime import datetime

# read in input files

In [ ]:
pgs = pd.read_csv('aou_ukbb_meta_prs/plink_score_output/concatenated_outputs/ADSP_keep_quest_comb.all_computed_PGS_scores.txt',
                  sep = '\t')
print(pgs.shape)
pgs.head()

In [ ]:
adsp_pheno = pd.read_csv('/project/ritchie/projects/ADSP_Projects/QC_ADSPv11/ADSPphenotype_forAnalysis.txt',
                         sep = '\t')
print(len(adsp_pheno.index))
adsp_pheno.head()

In [ ]:
omics_avg_pathway = pd.read_csv('pathway_score/rosmap/avg_pathways/ADSP.multiomics.pathway_average.pathway_scores.pathway_intersection.standard_scaled.go.keep_quest_comb.txt',
                                sep = '\t').transpose()
print(omics_avg_pathway.shape)
omics_avg_pathway.head()

In [ ]:
id_map = pd.read_csv('pathway_score/id_map/ADSP.ROSMAP.MSBB.id_map.txt',
                      sep = '\t')
id_map.head()

# clean files

## subset & rename

In [ ]:
pgs_sub = pgs[['#IID', 'EUR.AD.SCORE_AVG']].rename(columns = {'#IID' : 'IID'})
pgs_sub.head()

In [ ]:
adsp_pheno_sub = adsp_pheno[['IID', 'DX_harmonized', 'Age_harmonized', 'Sex', 'PC1', 'PC2', 'PC3', 'PC4']]
adsp_pheno_sub.head()

## merge & remove missing

In [ ]:
pgs_pheno_covar = pgs_sub.merge(adsp_pheno_sub, on = 'IID').dropna()
print(len(pgs_pheno_covar.index))
pgs_pheno_covar.head()

## add common ID

In [ ]:
id_map_sub = id_map[['CommonID', 'SampleID']].rename(columns = {'SampleID' : 'IID'})
id_map_sub.head()

In [ ]:
pgs_pheno_covar_common_id = pgs_pheno_covar.merge(id_map_sub, on = 'IID')
print(len(pgs_pheno_covar_common_id.index))
pgs_pheno_covar_common_id.head()

## remove omics individuals

In [ ]:
pgs_pheno_covar_sub = pgs_pheno_covar_common_id[~pgs_pheno_covar_common_id['CommonID'].isin(omics_avg_pathway.index)]
print(len(pgs_pheno_covar_sub.index))

# run PRS regressions

In [ ]:
# make lists
auroc_list = []
auprc_list = []
f1_list = []
ba_list = []

for iter in list(range(1, 101)):

    # split into train and test
    train = pgs_pheno_covar_sub.sample(frac = 0.7, random_state = iter)
    test = pgs_pheno_covar_sub.drop(train.index)

    # make predictors list
    predictors = train.columns.tolist()
    predictors.remove('IID')
    predictors.remove('CommonID')
    predictors.remove('DX_harmonized')

    # scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train[predictors])
    X_test_scaled = scaler.transform(test[predictors])

    # make base models
    ridge = LogisticRegression(penalty = 'l2', max_iter = 5000, n_jobs  = -1, class_weight = 'balanced', random_state = iter)

    # train
    model = ridge.fit(X_train_scaled, train['DX_harmonized'])

    # test
    y_test_pred_bin = model.predict(X_test_scaled)
    y_test_pred_cont = model.predict_proba(X_test_scaled)[:, 1]

    # compute metrics
    auroc = roc_auc_score(test['DX_harmonized'], y_test_pred_cont)
    auprc = average_precision_score(test['DX_harmonized'], y_test_pred_cont)
    f1 = f1_score(test['DX_harmonized'], y_test_pred_bin)
    ba = balanced_accuracy_score(test['DX_harmonized'], y_test_pred_bin)

    # append to lists
    auroc_list.append(auroc)
    auprc_list.append(auprc)
    f1_list.append(f1)
    ba_list.append(ba)

# check list lengths
print(len(auroc_list))
print(len(auprc_list))
print(len(f1_list))
print(len(ba_list))

# compute means
mean_auroc = np.mean(auroc_list)
mean_auprc = np.mean(auprc_list)
mean_f1 = np.mean(f1_list)
mean_ba = np.mean(ba_list)

# make df
metrics_df = pd.DataFrame({'MODEL' : ['PRScsx'], 'AUROC' : [mean_auroc], 'AUPRC' : [mean_auprc], 'F1_SCORE' : [mean_f1], 'BALANCED_ACCURACY' : [mean_ba]})
metrics_df

# run regressions- covars only

In [ ]:
# make lists
auroc_list = []
auprc_list = []
f1_list = []
ba_list = []

for iter in list(range(1, 101)):

    # split into train and test
    train = pgs_pheno_covar_sub.sample(frac = 0.7, random_state = iter)
    test = pgs_pheno_covar_sub.drop(train.index)

    # make predictors list
    predictors = train.columns.tolist()
    predictors.remove('IID')
    predictors.remove('CommonID')
    predictors.remove('DX_harmonized')
    predictors.remove('EUR.AD.SCORE_AVG')

    # scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train[predictors])
    X_test_scaled = scaler.transform(test[predictors])

    # make base models
    ridge = LogisticRegression(penalty = 'l2', max_iter = 5000, n_jobs  = -1, class_weight = 'balanced', random_state = iter)

    # train
    model = ridge.fit(X_train_scaled, train['DX_harmonized'])

    # test
    y_test_pred_bin = model.predict(X_test_scaled)
    y_test_pred_cont = model.predict_proba(X_test_scaled)[:, 1]

    # compute metrics
    auroc = roc_auc_score(test['DX_harmonized'], y_test_pred_cont)
    auprc = average_precision_score(test['DX_harmonized'], y_test_pred_cont)
    f1 = f1_score(test['DX_harmonized'], y_test_pred_bin)
    ba = balanced_accuracy_score(test['DX_harmonized'], y_test_pred_bin)

    # append to lists
    auroc_list.append(auroc)
    auprc_list.append(auprc)
    f1_list.append(f1)
    ba_list.append(ba)

# check list lengths
print(len(auroc_list))
print(len(auprc_list))
print(len(f1_list))
print(len(ba_list))

# compute means
mean_auroc = np.mean(auroc_list)
mean_auprc = np.mean(auprc_list)
mean_f1 = np.mean(f1_list)
mean_ba = np.mean(ba_list)

# make df
covar_metrics_df = pd.DataFrame({'MODEL' : ['Covariates'], 'AUROC' : [mean_auroc], 'AUPRC' : [mean_auprc], 'F1_SCORE' : [mean_f1], 'BALANCED_ACCURACY' : [mean_ba]})
covar_metrics_df

## export results df

In [ ]:
metrics_df.to_csv('aou_ukbb_meta_prs/ADSP_keep_quest_comb.PRScsx_covars.eval.csv', index = None)

In [ ]:
covar_metrics_df.to_csv('aou_ukbb_meta_prs/ADSP_keep_quest_comb.covars.eval.csv', index = None)